<a href="https://colab.research.google.com/github/Kun942ACGNV/0526/blob/main/smart_parking_lot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🅿️ 智慧停車場：以資料結構為架構

本 notebook 依「由核心到擴充」的順序，一步一個 cell 建構整個專案。
**使用方式：** 由上到下依序執行（`Runtime → Run all` 或逐格 `Shift+Enter`）。

| 步驟 | 功能 | 資料結構 |
|---|---|---|
| 1 | 基本資料物件 | `dataclass` |
| 2 | 車位管理、分配最近空位 | 二維陣列 + 最小堆積 (Heap) |
| 3 | 車牌前綴搜尋 | 字典樹 (Trie) |
| 4 | 進出場、計費 | 雜湊表 (dict) + 佇列 (deque) |
| 5 | 場內導航 | 圖 (Graph) + Dijkstra |
| 6 | 營收報表 | 有序串列 + 二分搜尋 (bisect) |
| 7 | 整合測試 | assert |
| 8 | 一日流量模擬與視覺化 | 以上全部 + matplotlib |
| 9 | 狀態存檔 / 讀檔 | JSON |
| 10 | 複雜度總結 | — |

## 步驟 0：環境與匯入
Colab 已內建 `matplotlib`、`networkx`，不需另外安裝。

In [ ]:
from dataclasses import dataclass
from collections import deque
import heapq, bisect, math, random, json, time

import matplotlib.pyplot as plt
import networkx as nx

# 中文字型（Colab 預設沒有中文字型，圖表標題用英文以避免亂碼）
print("環境就緒")

## 步驟 1：基本資料物件
- `Slot`：一個車位。`distance` 是離入口的距離，之後作為堆積的排序鍵。
- `Ticket`：一張停車票券，記錄車牌、車位、進場時間（秒）。

In [ ]:
@dataclass
class Slot:
    floor: int
    index: int
    distance: float
    kind: str = "normal"          # normal / ev / disabled
    plate: str | None = None

    @property
    def name(self) -> str:
        return f"F{self.floor}-{self.index}"

@dataclass
class Ticket:
    plate: str
    slot: Slot
    enter_time: float             # 秒（可用模擬時間或 time.time()）

print(Slot(0, 1, 11.0), Ticket("TEST-001", Slot(0, 1, 11.0), 0.0), sep="\n")

## 步驟 2：車位管理（二維陣列 + 最小堆積）
- **二維陣列** `slots[floor][index]`：依索引 O(1) 存取車位。
- **最小堆積** `free_heap`：存放空位 `(distance, floor, index)`，取最近空位 O(log n)。

In [ ]:
class SlotManager:
    def __init__(self, floors: int, per_floor: int, distance_map: dict | None = None):
        self.floors, self.per_floor = floors, per_floor
        self.slots = [
            [Slot(f, i, distance_map[f"F{f}-{i}"] if distance_map else f * 100 + i)
             for i in range(per_floor)]
            for f in range(floors)
        ]
        self.rebuild_heap()

    def rebuild_heap(self):
        self.free_heap = [(s.distance, s.floor, s.index)
                          for row in self.slots for s in row if s.plate is None]
        heapq.heapify(self.free_heap)

    def allocate(self) -> Slot | None:
        if not self.free_heap:
            return None
        _, f, i = heapq.heappop(self.free_heap)
        return self.slots[f][i]

    def release(self, slot: Slot):
        slot.plate = None
        heapq.heappush(self.free_heap, (slot.distance, slot.floor, slot.index))

    def free_count(self) -> int:
        return len(self.free_heap)

    def total(self) -> int:
        return self.floors * self.per_floor

# 測試
m = SlotManager(2, 3)
a, b = m.allocate(), m.allocate()
print("分配到：", a.name, b.name, "| 剩餘：", m.free_count())
m.release(a)
print("釋放 F0-0 後再分配：", m.allocate().name)

## 步驟 3：車牌搜尋（Trie 字典樹）
忘記車停哪時，輸入部分車牌就能列出候選。支援插入、前綴搜尋、刪除。

In [ ]:
class TrieNode:
    def __init__(self):
        self.children: dict[str, "TrieNode"] = {}
        self.is_end = False

class PlateTrie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, plate: str):
        node = self.root
        for ch in plate:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def search_prefix(self, prefix: str) -> list[str]:
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return []
            node = node.children[ch]
        result = []
        def dfs(n, path):
            if n.is_end:
                result.append(prefix + path)
            for c in sorted(n.children):
                dfs(n.children[c], path + c)
        dfs(node, "")
        return result

    def delete(self, plate: str):
        # 回傳 True 表示該節點可從父節點移除
        def _del(node, i):
            if i == len(plate):
                if not node.is_end:
                    return False
                node.is_end = False
                return not node.children
            child = node.children.get(plate[i])
            if child is None:
                return False
            if _del(child, i + 1):
                del node.children[plate[i]]
                return not node.children and not node.is_end
            return False
        _del(self.root, 0)

# 測試
t = PlateTrie()
for p in ["ABC-123", "ABC-456", "ABD-555", "XYZ-789"]:
    t.insert(p)
print("ABC 開頭：", t.search_prefix("ABC"))
t.delete("ABC-123")
print("刪除後：  ", t.search_prefix("ABC"))

## 步驟 4：進出場與計費（雜湊表 + 佇列）
- **佇列 `entry_queue`**：入口先到先服務（FIFO）。
- **雜湊表 `active`**：車牌 → 票券，查詢與出場 O(1)。
- **集合 `queued`**：避免同一車牌重複排隊，O(1) 判斷。
- 計費：每小時 30 元，不足一小時以一小時計，最低一小時。
- 所有時間參數 `now` 可手動傳入，方便做模擬與測試。

In [ ]:
class ParkingLot:
    RATE_PER_HOUR = 30

    def __init__(self, floors=3, per_floor=20, distance_map=None):
        self.mgr = SlotManager(floors, per_floor, distance_map)
        self.active: dict[str, Ticket] = {}
        self.entry_queue: deque[str] = deque()
        self.queued: set[str] = set()
        self.history: list[tuple[float, str, int]] = []   # (出場時間, 車牌, 費用)
        self.trie = PlateTrie()

    # --- 入口 ---
    def arrive(self, plate: str):
        if plate in self.active or plate in self.queued:
            raise ValueError(f"{plate} 已在場內或排隊中")
        self.entry_queue.append(plate)
        self.queued.add(plate)

    def process_entry(self, now: float | None = None) -> Ticket | None:
        now = time.time() if now is None else now
        if not self.entry_queue:
            return None
        slot = self.mgr.allocate()
        if slot is None:
            return None                     # 滿場：車輛繼續排隊
        plate = self.entry_queue.popleft()
        self.queued.discard(plate)
        slot.plate = plate
        ticket = Ticket(plate, slot, now)
        self.active[plate] = ticket
        self.trie.insert(plate)
        return ticket

    # --- 出口 ---
    def exit(self, plate: str, now: float | None = None) -> int:
        now = time.time() if now is None else now
        ticket = self.active.pop(plate)     # 不在場內會拋出 KeyError
        hours = max(1, math.ceil((now - ticket.enter_time) / 3600))
        fee = hours * self.RATE_PER_HOUR
        self.mgr.release(ticket.slot)
        self.trie.delete(plate)
        self.history.append((now, plate, fee))
        return fee

    # --- 查詢 ---
    def find(self, plate: str) -> str | None:
        t = self.active.get(plate)
        return t.slot.name if t else None

    def search(self, prefix: str) -> list[tuple[str, str]]:
        return [(p, self.find(p)) for p in self.trie.search_prefix(prefix)]

    def status(self) -> dict:
        return {"total": self.mgr.total(),
                "occupied": len(self.active),
                "free": self.mgr.free_count(),
                "waiting": len(self.entry_queue)}

# 測試
lot = ParkingLot(2, 3)
for p in ["ABC-123", "ABC-456", "XYZ-789"]:
    lot.arrive(p)
    tk = lot.process_entry(now=0)
    print(f"{p} → {tk.slot.name}")
print(lot.status())
print("搜尋 ABC：", lot.search("ABC"))
print("出場費用：", lot.exit("ABC-123", now=2.5 * 3600), "元（2.5 小時 → 計 3 小時）")

## 步驟 5：場內導航（圖 + Dijkstra）
把入口、路口、坡道、車位當作**節點**，通道當作有權重的**邊**。
Dijkstra 算出入口到每個車位的**實際行車距離**，再回頭用來決定車位的優先順序（取代固定的 `distance`）。

In [ ]:
def build_lot_graph(floors=2, per_floor=3):
    g = {}
    def add(u, v, w):
        g.setdefault(u, []).append((v, w))
        g.setdefault(v, []).append((u, w))
    add("入口", "路口F0", 5)
    for i in range(per_floor):
        add("路口F0", f"F0-{i}", 2 + 2 * i)
    prev = "路口F0"
    for f in range(1, floors):
        add(prev, f"路口F{f}", 10)          # 坡道
        for i in range(per_floor):
            add(f"路口F{f}", f"F{f}-{i}", 2 + 2 * i)
        prev = f"路口F{f}"
    return g

def dijkstra(graph, start):
    dist, prev = {start: 0}, {}
    pq = [(0, start)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist.get(u, math.inf):
            continue
        for v, w in graph[u]:
            nd = d + w
            if nd < dist.get(v, math.inf):
                dist[v], prev[v] = nd, u
                heapq.heappush(pq, (nd, v))
    return dist, prev

def shortest_path(graph, start, goal):
    dist, prev = dijkstra(graph, start)
    path, cur = [goal], goal
    while cur != start:
        cur = prev[cur]
        path.append(cur)
    return path[::-1], dist[goal]

G = build_lot_graph(2, 3)
path, d = shortest_path(G, "入口", "F1-2")
print("入口 → F1-2：", " → ".join(path), f"（距離 {d}）")

### 路網視覺化
（圖中節點使用代號，避免 Colab 缺少中文字型而顯示成方框。）

In [ ]:
def draw_graph(g, highlight_path=None):
    nxg = nx.Graph()
    for u, nbrs in g.items():
        for v, w in nbrs:
            nxg.add_edge(u, v, weight=w)
    # 中文節點改用英文代號顯示
    rename = {"入口": "ENTRANCE"}
    rename.update({n: n.replace("路口", "JCT-") for n in nxg.nodes if n.startswith("路口")})
    nxg = nx.relabel_nodes(nxg, rename)
    pos = nx.spring_layout(nxg, seed=7)
    plt.figure(figsize=(8, 5))
    nx.draw(nxg, pos, with_labels=True, node_color="#cfe8ff",
            node_size=1100, font_size=8)
    nx.draw_networkx_edge_labels(nxg, pos,
        edge_labels=nx.get_edge_attributes(nxg, "weight"), font_size=8)
    if highlight_path:
        hp = [rename.get(n, n.replace("路口", "JCT-")) for n in highlight_path]
        nx.draw_networkx_edges(nxg, pos, edgelist=list(zip(hp, hp[1:])),
                               edge_color="red", width=3)
    plt.title("Parking lot road network (red = shortest path)")
    plt.show()

draw_graph(G, path)

### 以實際距離分配車位
把 Dijkstra 的結果轉成 `distance_map`，再建立停車場。此後系統會優先分配**開車最近**的空位。

In [ ]:
dist_all, _ = dijkstra(G, "入口")
distance_map = {k: v for k, v in dist_all.items() if k.startswith("F") and "-" in k}
print(distance_map)

lot2 = ParkingLot(2, 3, distance_map=distance_map)
order = []
for i in range(6):
    lot2.arrive(f"CAR-{i}")
    order.append(lot2.process_entry(now=0).slot.name)
print("分配順序：", order)

## 步驟 6：營收報表（有序串列 + 二分搜尋）
`history` 依出場時間天然遞增，區間查詢用 `bisect` 做到 O(log n) 定位 + O(k) 加總。

In [ ]:
def revenue_between(history, t1, t2):
    times = [h[0] for h in history]
    lo = bisect.bisect_left(times, t1)
    hi = bisect.bisect_right(times, t2)
    return sum(h[2] for h in history[lo:hi]), hi - lo

# 測試
demo = ParkingLot(1, 5)
for i, hrs in enumerate([1, 2, 3, 4]):
    demo.arrive(f"R-{i}")
    demo.process_entry(now=0)
for i, hrs in enumerate([1, 2, 3, 4]):
    demo.exit(f"R-{i}", now=hrs * 3600)
rev, n = revenue_between(demo.history, 0, 2.5 * 3600)
print(f"前 2.5 小時內出場：{n} 輛，營收 {rev} 元")

## 步驟 7：整合測試
用 `assert` 驗證各資料結構的行為；全部通過會印出 ✅。

In [ ]:
lot = ParkingLot(2, 2)

# 1. 滿場時車輛排隊
for p in ["A1", "A2", "A3", "A4", "A5"]:
    lot.arrive(p)
while lot.process_entry(now=0):
    pass
assert lot.status() == {"total": 4, "occupied": 4, "free": 0, "waiting": 1}

# 2. 重複車牌會被拒絕
try:
    lot.arrive("A1");  assert False
except ValueError:
    pass
try:
    lot.arrive("A5");  assert False      # 正在排隊中
except ValueError:
    pass

# 3. 出場後，排隊車輛可進場，並重用被釋放的最近車位
freed_slot = lot.active["A1"].slot.name
assert lot.exit("A1", now=1800) == 30    # 30 分鐘 → 1 小時 → 30 元
assert lot.process_entry(now=1800).slot.name == freed_slot
assert lot.find("A5") == freed_slot

# 4. 出場車牌從 Trie 移除
assert "A1" not in lot.trie.search_prefix("A")

# 5. 不在場內的車牌出場會報錯
try:
    lot.exit("NOPE");  assert False
except KeyError:
    pass

print("✅ 全部測試通過")

## 步驟 8：一日流量模擬與視覺化
每 5 分鐘一個時間點，早晚尖峰到達率較高，隨機出場。繪出車位使用量與排隊長度。

In [ ]:
random.seed(42)
sim = ParkingLot(floors=3, per_floor=10, distance_map=None)
usage, waiting, hours_axis = [], [], []
car_id = 0

for minute in range(0, 24 * 60, 5):
    now = minute * 60
    h = minute / 60
    rate = 0.7 if (8 <= h < 10 or 17 <= h < 19) else 0.15
    for _ in range(2):                           # 每個時間點最多 2 輛到達
        if random.random() < rate:
            car_id += 1
            sim.arrive(f"SIM-{car_id:04d}")
    while sim.process_entry(now=now):
        pass
    for plate in list(sim.active):               # 每輛車每 5 分鐘有 4% 機率離場
        if random.random() < 0.04:
            sim.exit(plate, now=now)
    usage.append(len(sim.active))
    waiting.append(len(sim.entry_queue))
    hours_axis.append(h)

total_rev = sum(r[2] for r in sim.history)
print(f"到達 {car_id} 輛、完成出場 {len(sim.history)} 輛、營收 {total_rev} 元、最終排隊 {len(sim.entry_queue)} 輛")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hours_axis, usage, label="Occupied slots")
ax.plot(hours_axis, waiting, label="Waiting queue", linestyle="--")
ax.axhline(sim.mgr.total(), color="gray", linestyle=":", label="Capacity")
ax.set_xlabel("Hour of day"); ax.set_ylabel("Cars")
ax.set_title("One-day simulation"); ax.legend(); ax.grid(alpha=.3)
plt.show()

## 步驟 9：狀態存檔 / 讀檔（JSON）
Colab 的變數在執行階段結束後會消失，把狀態寫成 JSON 才能保存。
下面先存在 Colab 暫存空間；若要長期保存，取消註解掛載 Google Drive 的兩行，把路徑改到 Drive。

In [ ]:
def save_state(lot: ParkingLot, path: str):
    data = {
        "floors": lot.mgr.floors, "per_floor": lot.mgr.per_floor,
        "active": {p: {"slot": t.slot.name, "enter_time": t.enter_time}
                   for p, t in lot.active.items()},
        "queue": list(lot.entry_queue),
        "history": lot.history,
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def load_state(path: str, distance_map=None) -> ParkingLot:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    lot = ParkingLot(data["floors"], data["per_floor"], distance_map)
    for plate, info in data["active"].items():
        f, i = info["slot"][1:].split("-")
        slot = lot.mgr.slots[int(f)][int(i)]
        slot.plate = plate
        lot.active[plate] = Ticket(plate, slot, info["enter_time"])
        lot.trie.insert(plate)
    lot.mgr.rebuild_heap()
    for plate in data["queue"]:
        lot.entry_queue.append(plate); lot.queued.add(plate)
    lot.history = [tuple(h) for h in data["history"]]
    return lot

# from google.colab import drive
# drive.mount("/content/drive")      # 若要存到 Drive，路徑改為 /content/drive/MyDrive/parking.json

save_state(sim, "parking_state.json")
restored = load_state("parking_state.json")
assert restored.status() == sim.status()
assert sorted(restored.active) == sorted(sim.active)
print("✅ 存檔 / 讀檔一致：", restored.status())

## 步驟 10：複雜度總結

| 操作 | 資料結構 | 時間複雜度 |
|---|---|---|
| 分配最近空位 | 最小堆積 | O(log n) |
| 釋放車位 | 最小堆積 | O(log n) |
| 入口排隊 / 出隊 | deque | O(1) |
| 以車牌查車位 | dict | O(1) 平均 |
| 車牌前綴搜尋 | Trie | O(L + k)，L 為前綴長度，k 為結果字元數 |
| 最短路徑 | Dijkstra（堆積版） | O((V + E) log V) |
| 營收區間查詢 | bisect | O(log n) 定位（本實作建立 times 串列為 O(n)） |

### 可繼續擴充
- 預約車位：`dict` 存預約，另用最小堆積依到期時間自動釋放
- 電動車 / 身障車位：每種類型各一個堆積
- 月租車優先入場：把 `deque` 換成優先佇列
- 網頁版：核心邏輯可移植成 JavaScript 後放 GitHub Pages，或以 Flask 提供 API